# MatanglaWIN — Drone Crack Geotagging Demo (Colab)

Run crack **instance segmentation** on drone photos, convert the detected
polygons to **WGS84 (EPSG:4326)**, export **GeoJSON**, and view the cracks
on an interactive **Folium/Leaflet** map — all inline.

Georeferencing uses either an **orthomosaic GeoTIFF** transform (most
accurate) or, for individual photos, a **Ground Sample Distance** transform
from the flight altitude + camera specs (nadir / flat-ground approximation).

All logic lives in `geotag_engine.py` / `exif_metadata.py` — this notebook
only orchestrates it.

## 1. Install dependencies (Colab)

In [ ]:
# Core model stack + optional geospatial extras.
!pip -q install ultralytics opencv-python-headless numpy exifread pyproj folium
# rasterio is only needed if you georeference against an orthomosaic GeoTIFF:
# !pip -q install rasterio

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Get the repo + configure paths

Point `REPO_DIR` at your checkout (or clone it), and set the folder of
drone photos plus your camera parameters.

In [ ]:
import os, sys, glob

# Option A: clone the repo
# !git clone https://github.com/nstfrim2026-art/Matanglawin-2.0.git /content/Matanglawin-2.0
REPO_DIR = '/content/Matanglawin-2.0'
sys.path.insert(0, REPO_DIR)

WEIGHTS  = os.path.join(REPO_DIR, 'best.pt')                 # trained YOLO11-seg
IMAGE_DIR = '/content/drive/MyDrive/matanglawin/drone_photos'  # your photos
OUT_GEOJSON = '/content/drive/MyDrive/matanglawin/cracks.geojson'
ORTHOMOSAIC = None   # e.g. '/content/drive/MyDrive/matanglawin/ortho.tif'

# Camera specs for the GSD method (use your aircraft's real datasheet values).
from geotag_engine import CameraModel
CAMERA = CameraModel(sensor_width_mm=6.4, focal_length_mm=4.7, name='my-drone')
# If the photos have no altitude in EXIF/XMP, set the flight height (m):
ALTITUDE_M = None

images = sorted(glob.glob(os.path.join(IMAGE_DIR, '*.[jJ][pP][gG]')) +
                glob.glob(os.path.join(IMAGE_DIR, '*.[jJ][pP][eE][gG]')))
print(f'{len(images)} images found')

## 4. Run segmentation + geotagging → GeoJSON

In [ ]:
from geotag_engine import geotag_batch, write_geojson

fc = geotag_batch(
    images,
    weights=WEIGHTS,
    camera=CAMERA,
    altitude_m=ALTITUDE_M,
    orthomosaic=ORTHOMOSAIC,
)
write_geojson(fc, OUT_GEOJSON)

props = fc['properties']
print('cracks:', props['num_cracks'], '| images:', props.get('images_processed'))
if props.get('errors'):
    print('errors:', props['errors'])
print('saved:', OUT_GEOJSON)

## 5. Interactive map (Folium / Leaflet)

Georeferenced crack polygons are drawn on a satellite basemap with popups
showing area / length / confidence. Images without GPS are reported and
skipped on the map (their pixel-space result is still in the GeoJSON).

In [ ]:
import folium

geo_feats = [f for f in fc['features'] if f.get('geometry')]
print(f'{len(geo_feats)} georeferenced / {len(fc["features"])} total')

# Centre the map on the first georeferenced vertex, else a default.
center = [14.5995, 120.9842]
if geo_feats:
    lon, lat = geo_feats[0]['geometry']['coordinates'][0][0]
    center = [lat, lon]

m = folium.Map(location=center, zoom_start=20, max_zoom=24,
               tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
               attr='Esri World Imagery')

def popup_html(p):
    return (f"<b>{p.get('image_name','')}</b><br>"
            f"area: {p.get('crack_area_m2')} m²<br>"
            f"length: {p.get('length_m')} m<br>"
            f"confidence: {p.get('confidence')}<br>"
            f"time: {p.get('timestamp')}")

for f in geo_feats:
    ring = f['geometry']['coordinates'][0]
    latlon = [[pt[1], pt[0]] for pt in ring]  # folium wants [lat, lon]
    folium.Polygon(latlon, color='red', weight=2, fill=True, fill_opacity=0.4,
                   popup=folium.Popup(popup_html(f['properties']), max_width=260)).add_to(m)
    folium.CircleMarker(latlon[0], radius=3, color='red').add_to(m)

m

## Notes & limitations

* The **GSD method** assumes a near-nadir camera over locally flat ground;
  oblique gimbal pitch, terrain relief, and lens distortion add error. For
  survey-grade results, georeference against an **orthomosaic GeoTIFF**
  (`ORTHOMOSAIC = '.../ortho.tif'`, requires `rasterio`).
* Coordinates are exported in **WGS84 (EPSG:4326)**; `crack_area_m2` is
  geodesic when `pyproj` is installed.
* This is the offline analysis path — it does not affect the live web UI or
  the automatic photo-import workflow.